In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException  # Import ElementClickInterceptedException
import time

## Scrap The Artificial Plants

In [2]:
# URL to scrape
url = 'https://stocktrack.ca/?s=ikea&search=artificial%20plants'

# Initialize the Chrome WebDriver
driver = webdriver.Chrome()

# Navigate to the provided URL
driver.get(url)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 50)  # Set a timeout of 40 seconds
    # Locate the iframe by its tag name and switch to it
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")


start_dhx_f_id = 1
scraped_products = []
while True:
    try:
        # Wait for the products to load on the current page
        WebDriverWait(driver, 70).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
        
#         # Initialize a variable to track whether there are elements with dhx_f_id on the current page
        elements_found = False

        # Scrape the products on the current page
        for i in range(start_dhx_f_id, start_dhx_f_id + 5):
            div_xpath = f'//div[@dhx_f_id="{i}"]'

            for _ in range(5):
                try:
                    div_to_click = driver.find_element(By.XPATH, div_xpath)
                    if div_to_click:
                        div_to_click.click()
                    else:
                        break
                    div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

                    # Extract product information from the div element
                    image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
                    product_name = div_element.find_element(By.TAG_NAME, 'a').text
                    product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

                    # Split the text by line breaks to extract individual pieces of information
                    # Extract the text content from the parent element
                    product_info = div_element.text

                    # Split the text into lines and extract the relevant information
                    lines = product_info.split('\n')

                    # Initialize variables to store extracted information
                    sku = None
                    size = None
                    price = None

                    # Iterate through the lines to find relevant information
                    for line in lines:
                        if line.startswith("SKU:"):
                            sku = line.replace("SKU:", "").strip()
                        elif "cm" in line:
                            size = line.strip()
                        elif line.startswith("Price:"):
                            price = line.replace("Price:", "").strip()
                            
                    # Split the price into old and new price if applicable
                    if " " in price:
                        prices = price.split()
                        old_price = prices[0]  # First part is old price
                        new_price = prices[1]  # Second part is new price
                    else:
                        old_price = price
                        new_price = 'N/A'                        
                    # Print the extracted information
                    print(i)
                    print("image:", image)
                    print("Product Name :", product_name)
                    print("Product Link :", product_link)
                    print("SKU:", sku)
                    print("Size:", size)
                    print("Old Price:", old_price)
                    print("New Price:", new_price)
                    try:

                        stock_number_element = WebDriverWait(driver, 30).until(
                        EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
                        )
                        stock_number_coq = stock_number_element.text
                        print("Coquitlam Store Stock Number:", stock_number_coq)                        
                        coq_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[2]")
                        stock_prob_coq = coq_prob_element.text
                        print("Coquitlam Store Stock Probability:", stock_prob_coq)
                                         
                        rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
                        stock_number_rich = rich_element.text
                        print("Richmond Store Stock Number:", stock_number_rich)                       
                        rich_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[2]")
                        stock_prob_rich = rich_prob_element.text
                        print("Richmond Store Stock Probability:", stock_prob_rich)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '529')]/following-sibling::td[5]")
                        stock_number_halifax = stock_element.text
                        print("Halifax Store Stock Number:", stock_number_halifax)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '529')]/following-sibling::td[4]")
                        stock_prob_halifax = prob_element.text
                        print("Halifax Store Stock Probability:", stock_prob_halifax)   
                        

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '559')]/following-sibling::td[5]")
                        stock_number_quebec = stock_element.text
                        print("Quebec Store Stock Number:", stock_number_quebec)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '559')]/following-sibling::td[4]")
                        stock_prob_quebec = prob_element.text
                        print("Quebec Store Stock Probability:", stock_prob_quebec)                     

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '414')]/following-sibling::td[5]")
                        stock_number_bouch = stock_element.text
                        print("Boucherville Store Stock Number:", stock_number_bouch)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '414')]/following-sibling::td[4]")
                        stock_prob_bouch = prob_element.text
                        print("Boucherville Store Stock Probability:", stock_prob_bouch)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '039')]/following-sibling::td[5]")
                        stock_number_montreal = stock_element.text
                        print("Montreal Store Stock Number:", stock_number_montreal)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '039')]/following-sibling::td[4]")
                        stock_prob_montreal = prob_element.text
                        print("Montreal Store Stock Probability:", stock_prob_montreal)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '004')]/following-sibling::td[5]")
                        stock_number_ottawa = stock_element.text
                        print("Ottawa Store Stock Number:", stock_number_ottawa)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '004')]/following-sibling::td[4]")
                        stock_prob_ottawa = prob_element.text
                        print("Ottawa Store Stock Probability:", stock_prob_ottawa)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '149')]/following-sibling::td[5]")
                        stock_number_nyork = stock_element.text
                        print("North York Store Stock Number:", stock_number_nyork)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '149')]/following-sibling::td[4]")
                        stock_prob_nyork = prob_element.text
                        print("North York Store Stock Probability:", stock_prob_nyork)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '256')]/following-sibling::td[5]")
                        stock_number_etobicoke = stock_element.text
                        print("Etobicoke Store Stock Number:", stock_number_etobicoke)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '256')]/following-sibling::td[4]")
                        stock_prob_etobicoke = prob_element.text
                        print("Etobicoke Store Stock Probability:", stock_prob_etobicoke)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '372')]/following-sibling::td[5]")
                        stock_number_vaughan = stock_element.text
                        print("Vaughan Store Stock Number:", stock_number_vaughan)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '372')]/following-sibling::td[4]")
                        stock_prob_vaughan = prob_element.text
                        print("Vaughan Store Stock Probability:", stock_prob_vaughan)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '249')]/following-sibling::td[5]")
                        stock_number_winnipeg = stock_element.text
                        print("Winnipeg Store Stock Number:", stock_number_winnipeg)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '249')]/following-sibling::td[4]")
                        stock_prob_winnipeg = prob_element.text
                        print("Winnipeg Store Stock Probability:", stock_prob_winnipeg)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '040')]/following-sibling::td[5]")
                        stock_number_burlington = stock_element.text
                        print("Burlington Store Stock Number:", stock_number_burlington)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '040')]/following-sibling::td[4]")
                        stock_prob_burlington = prob_element.text
                        print("Burlington Store Stock Probability:", stock_prob_burlington)                          
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '349')]/following-sibling::td[5]")
                        stock_number_edmonton = stock_element.text
                        print("Edmonton Store Stock Number:", stock_number_edmonton)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '349')]/following-sibling::td[4]")
                        stock_prob_edmonton = prob_element.text
                        print("Edmonton Store Stock Probability:", stock_prob_edmonton)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '216')]/following-sibling::td[5]")
                        stock_number_calgary = stock_element.text
                        print("Calgary Store Stock Number:", stock_number_calgary)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '216')]/following-sibling::td[4]")
                        stock_prob_calgary = prob_element.text
                        print("Calgary Store Stock Probability:", stock_prob_calgary) 
                        
                        
                    except TimeoutException:
                        print("Timed out waiting for the Coquitlam stock number to load")
                    except NoSuchElementException:
                        print("Coquitlam stock number element not found.")       
                    print()
                    product_data = {
                    'image_url': image,
                    'product_name': product_name,
                    'product_link': product_link,
                    'product_size' : size,
                    'product_sku': sku,
                    'product_price_old': old_price,
                    'product_price_new' : new_price,
                    'stock_probability_coquitlam' : stock_prob_coq,   
                    'stock_number_coquitlam' : stock_number_coq,
                    'stock_probability_richmond' : stock_prob_rich,
                    'stock_number_richmond' : stock_number_rich,
                    'stock_probability_halifax' : stock_prob_halifax,    
                    'stock_number_halifax' : stock_number_halifax,
                    'stock_probability_quebec' : stock_prob_quebec,
                    'stock_number_quebec' : stock_number_quebec,
                    'stock_probability_boucherville' : stock_prob_bouch,
                    'stock_number_boucherville' : stock_number_bouch,
                    'stock_probability_montreal' : stock_prob_montreal,
                    'stock_number_montreal' : stock_number_montreal,
                    'stock_probability_ottawa' : stock_prob_ottawa,
                    'stock_number_ottawa' : stock_number_ottawa,
                    'stock_probability_nyork' : stock_prob_nyork,
                    'stock_number_nyork' : stock_number_nyork,
                    'stock_probability_etobicoke' : stock_prob_etobicoke,
                    'stock_number_etobicoke' : stock_number_etobicoke,
                    'stock_probability_vaughan' : stock_prob_vaughan,
                    'stock_number_vaughan' : stock_number_vaughan,
                    'stock_probability_burlington' : stock_prob_burlington,
                    'stock_number_burlington' : stock_number_burlington,
                    'stock_probability_winnipeg' : stock_prob_winnipeg,
                    'stock_number_winnipeg' : stock_number_winnipeg,
                    'stock_probability_edmonton' : stock_prob_edmonton,
                    'stock_number_edmonton' : stock_number_edmonton,
                    'stock_probability_calgary' : stock_prob_calgary,
                    'stock_number_calgary' : stock_number_calgary

                    }
                    # Append the product data to the list
                    scraped_products.append(product_data)
                    # Set elements_found to True since elements with dhx_f_id were found
                    elements_found = True

                    break  # Exit the loop if the click is successful
                except StaleElementReferenceException:
                    continue  # Retry if a StaleElementReferenceException occurs
                except ElementClickInterceptedException:
                    print("Element click intercepted. Trying again.")
        
        # If no elements with dhx_f_id were found on the current page, break out of the loop
        if not elements_found:
            break
        print()
        
        # Check if the "Next" button is clickable
        next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")
        if not next_page_link.is_enabled():
            break  # Break out of the loop if the "Next" button is not clickable

        # Move to the next page by clicking the 'next page' link
        ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()
        start_dhx_f_id += 5
    except TimeoutException:
        print("Timed out waiting for products to load.")
    
    except NoSuchElementException:
            print(f"Element with dhx_f_id='{i}' not found. Exiting loop.")
            break  # Exit the loop if the element is not found



# Close the WebDriver
# Print the scraped product data

time.sleep(10)
driver.quit()





1
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/
SKU: 10467804
Size: 23 cm (9 ")
Old Price: $69.99
New Price: N/A
Coquitlam Store Stock Number: 12
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 13
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 26
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 18
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 4
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 24
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 17
Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 11
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke S

Ottawa Store Stock Probability: LOW_IN_STOCK
North York Store Stock Number: 2
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 0
Etobicoke Store Stock Probability: OUT_OF_STOCK
Vaughan Store Stock Number: 0
Vaughan Store Stock Probability: OUT_OF_STOCK
Winnipeg Store Stock Number: 0
Winnipeg Store Stock Probability: OUT_OF_STOCK
Burlington Store Stock Number: 1
Burlington Store Stock Probability: LOW_IN_STOCK
Edmonton Store Stock Number: 10
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 13
Calgary Store Stock Probability: HIGH_IN_STOCK

7
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-hanging-eucalyptus__0817871_pe774216_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-hanging-eucalyptus-70466811/
SKU: 70466811
Size: 9 cm (3 ½ ")
Old Price: $7.99
New Price: N/A
Coquitlam Store Stock Numbe

Coquitlam Store Stock Number: 17
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 8
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 12
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 5
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 29
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 16
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 27
Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 6
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 25
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 12
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 24
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 7
Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 20
Edmo

Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 582
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 752
Calgary Store Stock Probability: HIGH_IN_STOCK

18
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-grass__0130933_pe285358_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-grass-00433942/
SKU: 00433942
Size: 9 cm (3 ½ ")
Old Price: $2.99
New Price: N/A
Coquitlam Store Stock Number: 378
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 471
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 598
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 442
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 506
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 267
Mo

Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 15
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 14
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 24
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 15
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 90
Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 11
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 0
Calgary Store Stock Probability: OUT_OF_STOCK

24
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-orchid-white__0748887_pe745276_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-orchid-white-00285908/
SKU: 00285908
Size: 9 cm (3 ½ ")
Old Price: $9.99
New Price: N/A
Coquitlam Store Stock Number: 70
Coquitlam Store Stock Proba

...

Winnipeg Store Stock Number: 66
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 0
Burlington Store Stock Probability: OUT_OF_STOCK
Edmonton Store Stock Number: 42
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 47
Calgary Store Stock Probability: HIGH_IN_STOCK

69
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-poppy-pink__1188145_pe899373_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-poppy-pink-30560151/
SKU: 30560151
Size: 27 cm (10 ¾ ")
Old Price: $1.49
New Price: N/A
Coquitlam Store Stock Number: 74
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK
Halifax Store Stock Number: 103
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 136
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store St

Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 22
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 64
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 70
Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 23
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 3
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 1
Vaughan Store Stock Probability: LOW_IN_STOCK
Winnipeg Store Stock Number: 11
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 0
Burlington Store Stock Probability: OUT_OF_STOCK
Edmonton Store Stock Number: 11
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 0
Calgary Store Stock Probability: OUT_OF_STOCK

75
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-lily-white__0636960_pe698122_s5.jpg
Product Name : SMYCKA, Artificial fl

Coquitlam Store Stock Number: 46
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 6
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 87
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 9
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 23
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 3
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 28
Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 39
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 11
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 38
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 28
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 63
Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 19
Edm

Calgary Store Stock Probability: HIGH_IN_STOCK


86
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-indoor-outdoor-poppy__1248048_pe922958_s5.jpg
Product Name : SMYCKA, Artificial bouquet
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-bouquet-indoor-outdoor-poppy-10571810/
SKU: 10571810
Size: 48 cm (19 ")
Old Price: $11.99
New Price: N/A
Coquitlam Store Stock Number: 1
Coquitlam Store Stock Probability: LOW_IN_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK
Halifax Store Stock Number: 34
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 1
Quebec Store Stock Probability: LOW_IN_STOCK
Boucherville Store Stock Number: 1
Boucherville Store Stock Probability: LOW_IN_STOCK
Montreal Store Stock Number: 23
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 10
Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 0
North York Store Stock Probabili

Ottawa Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 20
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 27
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 6
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 7
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 43
Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 34
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 18
Calgary Store Stock Probability: HIGH_IN_STOCK

92
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-wall-mounted-indoor-outdoor-green-lilac__1183252_pe897453_s5.jpg
Product Name : FEJKA, Artificial plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-plant-wall-mounted-indoor-outdoor-green-lilac-50546569/
SKU: 50546569
Size: 26x26 cm (10 ¼x10 ¼ ")
Old Price: $5.99
New Price: N/A
Coquitlam Store S

In [3]:
import csv
import datetime

# Get the current date
current_date = datetime.datetime.now()

# Format the date as YYYY_MM_DD
formatted_date = current_date.strftime("%Y_%m_%d")
formatted_date2 = current_date.strftime("%Y/%m/%d")

# Create a dynamic filename based on the current date
csv_file_path = f"products_artifical_plants_{formatted_date}.csv"

# Create or open the CSV file for writing
with open(csv_file_path, mode='w', newline='') as csv_file:
    # Define the CSV headers (column names)
    fieldnames = [
        'current_date',
        'image_url',
        'product_name',
        'product_link',
        'product_size',
        'product_sku',
        'product_price_old',
        'product_price_new',
        'stock_probability_coquitlam',
        'stock_number_coquitlam',
        'stock_probability_richmond',
        'stock_number_richmond',
        'stock_probability_halifax',
        'stock_number_halifax',
        'stock_probability_quebec',
        'stock_number_quebec',
        'stock_probability_boucherville',
        'stock_number_boucherville',
        'stock_probability_montreal',
        'stock_number_montreal',
        'stock_probability_ottawa',
        'stock_number_ottawa',
        'stock_probability_nyork',
        'stock_number_nyork',
        'stock_probability_etobicoke',
        'stock_number_etobicoke',
        'stock_probability_vaughan',
        'stock_number_vaughan',
        'stock_probability_burlington',
        'stock_number_burlington',
        'stock_probability_winnipeg',
        'stock_number_winnipeg',
        'stock_probability_edmonton',
        'stock_number_edmonton',
        'stock_probability_calgary',
        'stock_number_calgary'      
    ]

    # Create a CSV writer
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Write the header row to the CSV file
    csv_writer.writeheader()

    # Iterate through the scraped_products list and write each product's data
    for product in scraped_products:
        product['current_date'] = formatted_date2
        csv_writer.writerow(product)

print(f"CSV file has been created : '{csv_file_path}'.")

CSV file has been created : 'products_artifical_plants_2024_02_05.csv'.


### Scrap The Real Plants

In [7]:
# URL to scrape
url = 'https://stocktrack.ca/?s=ikea&search=real%20plants'

# Initialize the Chrome WebDriver
driver = webdriver.Chrome()

# Navigate to the provided URL
driver.get(url)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 50)  # Set a timeout of 40 seconds
    # Locate the iframe by its tag name and switch to it
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")


start_dhx_f_id = 1
scraped_products = []
while True:
    try:
        # Wait for the products to load on the current page
        WebDriverWait(driver, 60).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
        
#         # Initialize a variable to track whether there are elements with dhx_f_id on the current page
        elements_found = False

        # Scrape the products on the current page
        for i in range(start_dhx_f_id, start_dhx_f_id + 5):
            div_xpath = f'//div[@dhx_f_id="{i}"]'

            for _ in range(5):
                try:
                    div_to_click = driver.find_element(By.XPATH, div_xpath)
                    if div_to_click:
                        div_to_click.click()
                    else:
                        break
                    div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

                    # Extract product information from the div element
                    image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
                    product_name = div_element.find_element(By.TAG_NAME, 'a').text
                    product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

                    # Split the text by line breaks to extract individual pieces of information
                    # Extract the text content from the parent element
                    product_info = div_element.text

                    # Split the text into lines and extract the relevant information
                    lines = product_info.split('\n')

                    # Initialize variables to store extracted information
                    sku = None
                    size = None
                    price = None

                    # Iterate through the lines to find relevant information
                    for line in lines:
                        if line.startswith("SKU:"):
                            sku = line.replace("SKU:", "").strip()
                        elif "cm" in line:
                            size = line.strip()
                        elif line.startswith("Price:"):
                            price = line.replace("Price:", "").strip()
                            
                    # Split the price into old and new price if applicable
                    if " " in price:
                        prices = price.split()
                        old_price = prices[0]  # First part is old price
                        new_price = prices[1]  # Second part is new price
                    else:
                        old_price = price
                        new_price = 'N/A'                        
                    # Print the extracted information
                    print(i)
                    print("image:", image)
                    print("Product Name :", product_name)
                    print("Product Link :", product_link)
                    print("SKU:", sku)
                    print("Size:", size)
                    print("Old Price:", old_price)
                    print("New Price:", new_price)
                    try:

                        stock_number_element = WebDriverWait(driver, 30).until(
                        EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
                        )
                        stock_number_coq = stock_number_element.text
                        print("Coquitlam Store Stock Number:", stock_number_coq)                        
                        coq_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[2]")
                        stock_prob_coq = coq_prob_element.text
                        print("Coquitlam Store Stock Probability:", stock_prob_coq)
                                         
                        rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
                        stock_number_rich = rich_element.text
                        print("Richmond Store Stock Number:", stock_number_rich)                       
                        rich_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[2]")
                        stock_prob_rich = rich_prob_element.text
                        print("Richmond Store Stock Probability:", stock_prob_rich)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '529')]/following-sibling::td[5]")
                        stock_number_halifax = stock_element.text
                        print("Halifax Store Stock Number:", stock_number_halifax)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '529')]/following-sibling::td[4]")
                        stock_prob_halifax = prob_element.text
                        print("Halifax Store Stock Probability:", stock_prob_halifax)   
                        

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '559')]/following-sibling::td[5]")
                        stock_number_quebec = stock_element.text
                        print("Quebec Store Stock Number:", stock_number_quebec)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '559')]/following-sibling::td[4]")
                        stock_prob_quebec = prob_element.text
                        print("Quebec Store Stock Probability:", stock_prob_quebec)                     

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '414')]/following-sibling::td[5]")
                        stock_number_bouch = stock_element.text
                        print("Boucherville Store Stock Number:", stock_number_bouch)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '414')]/following-sibling::td[4]")
                        stock_prob_bouch = prob_element.text
                        print("Boucherville Store Stock Probability:", stock_prob_bouch)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '039')]/following-sibling::td[5]")
                        stock_number_montreal = stock_element.text
                        print("Montreal Store Stock Number:", stock_number_montreal)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '039')]/following-sibling::td[4]")
                        stock_prob_montreal = prob_element.text
                        print("Montreal Store Stock Probability:", stock_prob_montreal)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '004')]/following-sibling::td[5]")
                        stock_number_ottawa = stock_element.text
                        print("Ottawa Store Stock Number:", stock_number_ottawa)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '004')]/following-sibling::td[4]")
                        stock_prob_ottawa = prob_element.text
                        print("Ottawa Store Stock Probability:", stock_prob_ottawa)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '659')]/following-sibling::td[5]")
                        stock_number_toronto = stock_element.text
                        print("Toronto Store Stock Number:", stock_number_toronto)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '659')]/following-sibling::td[4]")
                        stock_prob_toronto = prob_element.text
                        print("Toronto Store Stock Probability:", stock_prob_toronto)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '149')]/following-sibling::td[5]")
                        stock_number_nyork = stock_element.text
                        print("North York Store Stock Number:", stock_number_nyork)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '149')]/following-sibling::td[4]")
                        stock_prob_nyork = prob_element.text
                        print("North York Store Stock Probability:", stock_prob_nyork)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '256')]/following-sibling::td[5]")
                        stock_number_etobicoke = stock_element.text
                        print("Etobicoke Store Stock Number:", stock_number_etobicoke)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '256')]/following-sibling::td[4]")
                        stock_prob_etobicoke = prob_element.text
                        print("Etobicoke Store Stock Probability:", stock_prob_etobicoke)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '372')]/following-sibling::td[5]")
                        stock_number_vaughan = stock_element.text
                        print("Vaughan Store Stock Number:", stock_number_vaughan)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '372')]/following-sibling::td[4]")
                        stock_prob_vaughan = prob_element.text
                        print("Vaughan Store Stock Probability:", stock_prob_vaughan)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '249')]/following-sibling::td[5]")
                        stock_number_winnipeg = stock_element.text
                        print("Winnipeg Store Stock Number:", stock_number_winnipeg)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '249')]/following-sibling::td[4]")
                        stock_prob_winnipeg = prob_element.text
                        print("Winnipeg Store Stock Probability:", stock_prob_winnipeg)  

                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '040')]/following-sibling::td[5]")
                        stock_number_burlington = stock_element.text
                        print("Burlington Store Stock Number:", stock_number_burlington)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '040')]/following-sibling::td[4]")
                        stock_prob_burlington = prob_element.text
                        print("Burlington Store Stock Probability:", stock_prob_burlington)                          
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '349')]/following-sibling::td[5]")
                        stock_number_edmonton = stock_element.text
                        print("Edmonton Store Stock Number:", stock_number_edmonton)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '349')]/following-sibling::td[4]")
                        stock_prob_edmonton = prob_element.text
                        print("Edmonton Store Stock Probability:", stock_prob_edmonton)
                        
                        stock_element = driver.find_element(By.XPATH, "//td[contains(text(), '216')]/following-sibling::td[5]")
                        stock_number_calgary = stock_element.text
                        print("Calgary Store Stock Number:", stock_number_calgary)                       
                        prob_element = driver.find_element(By.XPATH, "//td[contains(text(), '216')]/following-sibling::td[4]")
                        stock_prob_calgary = prob_element.text
                        print("Calgary Store Stock Probability:", stock_prob_calgary) 
                        
                    except TimeoutException:
                        print("Timed out waiting for the Coquitlam stock number to load")
                    except NoSuchElementException:
                        print("Coquitlam stock number element not found.")       
                    print()
                    product_data = {
                    'image_url': image,
                    'product_name': product_name,
                    'product_link': product_link,
                    'product_size' : size,
                    'product_sku': sku,
                    'product_price_old': old_price,
                    'product_price_new' : new_price,
                    'stock_probability_coquitlam' : stock_prob_coq,   
                    'stock_number_coquitlam' : stock_number_coq,
                    'stock_probability_richmond' : stock_prob_rich,
                    'stock_number_richmond' : stock_number_rich,
                    'stock_probability_halifax' : stock_prob_halifax,    
                    'stock_number_halifax' : stock_number_halifax,
                    'stock_probability_quebec' : stock_prob_quebec,
                    'stock_number_quebec' : stock_number_quebec,
                    'stock_probability_boucherville' : stock_prob_bouch,
                    'stock_number_boucherville' : stock_number_bouch,
                    'stock_probability_montreal' : stock_prob_montreal,
                    'stock_number_montreal' : stock_number_montreal,
                    'stock_probability_ottawa' : stock_prob_ottawa,
                    'stock_probability_toronto' : stock_prob_toronto,                        
                    'stock_number_ottawa' : stock_number_ottawa,
                    'stock_probability_nyork' : stock_prob_nyork,
                    'stock_number_nyork' : stock_number_nyork,
                    'stock_probability_etobicoke' : stock_prob_etobicoke,
                    'stock_number_etobicoke' : stock_number_etobicoke,
                    'stock_probability_vaughan' : stock_prob_vaughan,
                    'stock_number_vaughan' : stock_number_vaughan,
                    'stock_probability_burlington' : stock_prob_burlington,
                    'stock_number_burlington' : stock_number_burlington,
                    'stock_probability_winnipeg' : stock_prob_winnipeg,
                    'stock_number_winnipeg' : stock_number_winnipeg,
                    'stock_probability_edmonton' : stock_prob_edmonton,
                    'stock_number_edmonton' : stock_number_edmonton,
                    'stock_probability_calgary' : stock_prob_calgary,
                    'stock_number_calgary' : stock_number_calgary

                    }
                    # Append the product data to the list
                    scraped_products.append(product_data)
                    # Set elements_found to True since elements with dhx_f_id were found
                    elements_found = True

                    break  # Exit the loop if the click is successful
                except StaleElementReferenceException:
                    continue  # Retry if a StaleElementReferenceException occurs
                except ElementClickInterceptedException:
                    print("Element click intercepted. Trying again.")
        
        # If no elements with dhx_f_id were found on the current page, break out of the loop
        if not elements_found:
            break
        print()
        
        # Check if the "Next" button is clickable
        next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")
        if not next_page_link.is_enabled():
            break  # Break out of the loop if the "Next" button is not clickable

        # Move to the next page by clicking the 'next page' link
        ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()
        start_dhx_f_id += 5
    except TimeoutException:
        print("Timed out waiting for products to load.")
    
    except NoSuchElementException:
            print(f"Element with dhx_f_id='{i}' not found. Exiting loop.")
            break  # Exit the loop if the element is not found



# Close the WebDriver
# Print the scraped product data

time.sleep(10)
driver.quit()





1
image: https://www.ikea.com/ca/en/images/products/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage__67453_pe181294_s5.jpg
Product Name : HIMALAYAMIX, Potted plant
Product Link : https://www.ikea.com/ca/en/p/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage-20197227/
SKU: 20197227
Size: 10 cm (4 ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 0
Coquitlam Store Stock Probability: OUT_OF_STOCK
Richmond Store Stock Number: 18
Richmond Store Stock Probability: MEDIUM_IN_STOCK
Halifax Store Stock Number: 108
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 78
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 108
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 68
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 104
Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 62
Toronto Store Stock Probability: H

Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 61
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 44
Calgary Store Stock Probability: HIGH_IN_STOCK

7
image: https://www.ikea.com/ca/en/images/products/monstera-deliciosa-potted-plant__0364397_pe547591_s5.jpg
Product Name : MONSTERA DELICIOSA, Potted plant
Product Link : https://www.ikea.com/ca/en/p/monstera-deliciosa-potted-plant-70555973/
SKU: 70555973
Size: 24 cm (9 ½ ")
Old Price: $39.99
New Price: N/A
Coquitlam Store Stock Number: 0
Coquitlam Store Stock Probability: OUT_OF_STOCK
Richmond Store Stock Number: 7
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 3
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 0
Quebec Store Stock Probability: OUT_OF_STOCK
Boucherville Store Stock Number: 9
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 1
Montreal Store Stock Probability: LOW_IN_STOCK
Ottawa

Ottawa Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.

14
image: https://www.ikea.com/ca/en/images/products/succulent-plant-with-pot-arrangement-assorted-species-plants__1169058_pe892337_s5.jpg
Product Name : SUCCULENT, Plant with pot
Product Link : https://www.ikea.com/ca/en/p/succulent-plant-with-pot-arrangement-assorted-species-plants-90493677/
SKU: 90493677
Size: 13 cm (5 ")
Old Price: $16.99
New Price: N/A
Coquitlam Store Stock Number: 15
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 17
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 24
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 23
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 24
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 25
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 9
Ottawa Store Stock Probability: HIGH_I

Ottawa Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.

22
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-monstera__0959226_pe809439_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-monstera-30493350/
SKU: 30493350
Size: 12 cm (4 ¾ ")
Old Price: $12.99
New Price: N/A
Coquitlam Store Stock Number: 78
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 45
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 35
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 75
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 71
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 77
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 164
Ottawa Store Stock Probability: HIGH_IN_STO

Coquitlam Store Stock Number: 66
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 77
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 24
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 30
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 35
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 137
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 52
Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 57
Toronto Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 42
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 244
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 25
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 69
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 405
B

...

Coquitlam Store Stock Number: 46
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 6
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 87
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 9
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 23
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 3
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 28
Ottawa Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.

129
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-rose-red__1248064_pe922968_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-rose-red-40571795/
SKU: 40571795
Size: 52 cm (20 ½ ")
Old Price: $2.99
New Price: N/A
Coquitlam Store Stock Number: 35
Coquitlam Store Stock Probab

Ottawa Store Stock Number: 205
Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 78
Toronto Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 581
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 162
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 704
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 199
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Number: 318
Burlington Store Stock Probability: HIGH_IN_STOCK
Edmonton Store Stock Number: 433
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 949
Calgary Store Stock Probability: HIGH_IN_STOCK


136
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-peony-white__0611399_pe685423_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-peony-white-80409783/
SKU: 80409783
Size: 30

Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 7
Toronto Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 2
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 0
Etobicoke Store Stock Probability: OUT_OF_STOCK
Vaughan Store Stock Number: 0
Vaughan Store Stock Probability: OUT_OF_STOCK
Winnipeg Store Stock Number: 0
Winnipeg Store Stock Probability: OUT_OF_STOCK
Burlington Store Stock Number: 1
Burlington Store Stock Probability: MEDIUM_IN_STOCK
Edmonton Store Stock Number: 17
Edmonton Store Stock Probability: HIGH_IN_STOCK
Calgary Store Stock Number: 2
Calgary Store Stock Probability: MEDIUM_IN_STOCK

143
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-eucalyptus-pink__0611429_pe685430_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-eucalyptus-pink-30409846/
SKU: 30409846
Size: 30 cm (11 ¾ ")
Old Price: $1.99
New Pri

148
image: https://www.ikea.com/ca/en/images/products/daksjus-hanging-planter-set-of-2-indoor-outdoor-light-gray-green__1221261_pe913680_s5.jpg
Product Name : DAKSJUS, Hanging planter, set of 2
Product Link : https://www.ikea.com/ca/en/p/daksjus-hanging-planter-set-of-2-indoor-outdoor-light-gray-green-10567035/
SKU: 10567035
Size: None
Old Price: $24.99
New Price: N/A
Coquitlam Store Stock Number: 21
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK
Halifax Store Stock Number: 9
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 14
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 1
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 15
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 6
Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 4
Toronto Store Stock Probability: HIGH_IN_STO

Coquitlam Store Stock Number: 378
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 471
Richmond Store Stock Probability: HIGH_IN_STOCK
Halifax Store Stock Number: 598
Halifax Store Stock Probability: HIGH_IN_STOCK
Quebec Store Stock Number: 442
Quebec Store Stock Probability: HIGH_IN_STOCK
Boucherville Store Stock Number: 506
Boucherville Store Stock Probability: HIGH_IN_STOCK
Montreal Store Stock Number: 267
Montreal Store Stock Probability: HIGH_IN_STOCK
Ottawa Store Stock Number: 712
Ottawa Store Stock Probability: HIGH_IN_STOCK
Toronto Store Stock Number: 167
Toronto Store Stock Probability: HIGH_IN_STOCK
North York Store Stock Number: 153
North York Store Stock Probability: HIGH_IN_STOCK
Etobicoke Store Stock Number: 184
Etobicoke Store Stock Probability: HIGH_IN_STOCK
Vaughan Store Stock Number: 429
Vaughan Store Stock Probability: HIGH_IN_STOCK
Winnipeg Store Stock Number: 477
Winnipeg Store Stock Probability: HIGH_IN_STOCK
Burlington Store Stock Num

In [ ]:
for product in scraped_products:
    print(product)
    print() 

In [ ]:
# Get the current date
current_date = datetime.datetime.now()

# Format the date as YYYY_MM_DD
formatted_date = current_date.strftime("%Y_%m_%d")
formatted_date2 = current_date.strftime("%Y/%m/%d")

# Create a dynamic filename based on the current date
csv_file_path = f"products_real_plants_{formatted_date}.csv"

# Create or open the CSV file for writing
with open(csv_file_path, mode='w', newline='') as csv_file:
    # Define the CSV headers (column names)
    fieldnames = [
        'current_date',
        'image_url',
        'product_name',
        'product_link',
        'product_size',
        'product_sku',
        'product_price_old',
        'product_price_new',
        'stock_probability_coquitlam',
        'stock_number_coquitlam',
        'stock_probability_richmond',
        'stock_number_richmond',
        'stock_probability_halifax',
        'stock_number_halifax',
        'stock_probability_quebec',
        'stock_number_quebec',
        'stock_probability_boucherville',
        'stock_number_boucherville',
        'stock_probability_montreal',
        'stock_number_montreal',
        'stock_probability_ottawa',
        'stock_number_ottawa',
        'stock_probability_nyork',
        'stock_number_nyork',
        'stock_probability_etobicoke',
        'stock_number_etobicoke',
        'stock_probability_vaughan',
        'stock_number_vaughan',
        'stock_probability_burlington',
        'stock_number_burlington',
        'stock_probability_winnipeg',
        'stock_number_winnipeg',
        'stock_probability_edmonton',
        'stock_number_edmonton',
        'stock_probability_calgary',
        'stock_number_calgary'      
    ]

    # Create a CSV writer
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Write the header row to the CSV file
    csv_writer.writeheader()

    # Iterate through the scraped_products list and write each product's data
    for product in scraped_products:
        product['current_date'] = formatted_date2
        csv_writer.writerow(product)

print(f"CSV file has been created : '{csv_file_path}'.")